# M4 Competition

This shows an example with just 4 series of the M4 dataset. If you want
to run it yourself on all of them, you can refer to [this
notebook](https://www.kaggle.com/code/lemuz90/m4-competition).

In [ ]:
import random

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from utilsforecast.plotting import plot_series

from mlforecast import MLForecast
from mlforecast.lag_transforms import (
    ExponentiallyWeightedMean,
    RollingMean,
)
from mlforecast.lgb_cv import LightGBMCV
from mlforecast.target_transforms import Differences
from mlforecast.utils import PredictionIntervals

In [ ]:
#| echo: false
import warnings

warnings.filterwarnings(action="ignore", category=FutureWarning)
warnings.filterwarnings(action="ignore", category=UserWarning)

In [ ]:
df = pd.read_parquet('https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet')
ids = df['unique_id'].unique()
random.seed(0)
sample_ids = random.choices(ids, k=4)
sample_df = df[df['unique_id'].isin(sample_ids)]
sample_df

We now split this data into train and validation.

In [ ]:
horizon = 48
valid = sample_df.groupby('unique_id').tail(horizon)
train = sample_df.drop(valid.index)
train.shape, valid.shape

## Creating the forecaster

In [ ]:
fcst = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq=1,
    lags=[24 * (i+1) for i in range(7)],
    lag_transforms={
        48: [ExponentiallyWeightedMean(alpha=0.3)],
    },
    num_threads=1,
    target_transforms=[Differences([24])],
)

In [ ]:
fcst

Once we have this setup we can compute the features and fit the model.

## Fitting and predicting

In [ ]:
fcst = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq=1,
    lags=[24 * (i+1) for i in range(7)],
    lag_transforms={
        48: [ExponentiallyWeightedMean(alpha=0.3)],
    },
    num_threads=1,
    target_transforms=[Differences([24])],
)

In [ ]:
train2 = train.copy()
train2['weight'] = np.random.default_rng(seed=0).random(train2.shape[0])
fcst.fit(train2, weight_col='weight', as_numpy=True).predict(5)

In [ ]:
fcst.cross_validation(train2, n_windows=2, h=5, weight_col='weight', as_numpy=True)

In [ ]:
fcst.fit(train, fitted=True);

In [ ]:
expected_future = fcst.make_future_dataframe(h=1)
expected_future

In [ ]:
missing_future = fcst.get_missing_future(h=1, X_df=expected_future.head(2))
pd.testing.assert_frame_equal(
    missing_future,
    expected_future.tail(2).reset_index(drop=True)
)

In [ ]:
fcst.forecast_fitted_values()

In [ ]:
fcst.forecast_fitted_values(level=[90])

Once we’ve run this we’re ready to compute our predictions.

In [ ]:
predictions = fcst.predict(horizon)

We can see at a couple of results.

In [ ]:
results = valid.merge(predictions, on=['unique_id', 'ds'])
fig = plot_series(forecasts_df=results)

In [ ]:
fig

### Prediction intervals

With [`MLForecast`](https://Nixtla.github.io/mlforecast/forecast.html#mlforecast),
you can generate prediction intervals using Conformal Prediction. To
configure Conformal Prediction, you need to pass an instance of the
[`PredictionIntervals`](https://Nixtla.github.io/mlforecast/utils.html#predictionintervals)
class to the `prediction_intervals` argument of the `fit` method. The
class takes three parameters: `n_windows`, `h` and `method`.

-   `n_windows` represents the number of cross-validation windows used
    to calibrate the intervals
-   `h` is the forecast horizon
-   `method` can be `conformal_distribution` or `conformal_error`;
    `conformal_distribution` (default) creates forecasts paths based on
    the cross-validation errors and calculate quantiles using those
    paths, on the other hand `conformal_error` calculates the error
    quantiles to produce prediction intervals. The strategy will adjust
    the intervals for each horizon step, resulting in different widths
    for each step. Please note that a minimum of 2 cross-validation
    windows must be used.

In [ ]:
fcst.fit(
    train,
    prediction_intervals=PredictionIntervals(n_windows=3, h=48)
)

After that, you just have to include your desired confidence levels to
the `predict` method using the `level` argument. Levels must lie between
0 and 100.

In [ ]:
predictions_w_intervals = fcst.predict(48, level=[50, 80, 95])
predictions_w_intervals.head()

Let’s explore the generated intervals.

In [ ]:
results = valid.merge(predictions_w_intervals, on=['unique_id', 'ds'])
fig = plot_series(forecasts_df=results, level=[50, 80, 95])
fig

If you want to reduce the computational time and produce intervals with
the same width for the whole forecast horizon, simple pass `h=1` to the
[`PredictionIntervals`](https://Nixtla.github.io/mlforecast/utils.html#predictionintervals)
class. The caveat of this strategy is that in some cases, variance of
the absolute residuals maybe be small (even zero), so the intervals may
be too narrow.

In [ ]:
fcst.fit(
    train,
    prediction_intervals=PredictionIntervals(n_windows=3, h=1)
);

In [ ]:
predictions_w_intervals_ws_1 = fcst.predict(48, level=[80, 90, 95])

Let’s explore the generated intervals.

In [ ]:
results = valid.merge(predictions_w_intervals_ws_1, on=['unique_id', 'ds'])
fig = plot_series(forecasts_df=results, level=[90])
fig

### Forecast using a pretrained model

MLForecast allows you to use a pretrained model to generate forecasts
for a new dataset. Simply provide a pandas dataframe containing the new
observations as the value for the `new_df` argument when calling the
`predict` method. The dataframe should have the same structure as the
one used to fit the model, including any features and time series data.
The function will then use the pretrained model to generate forecasts
for the new observations. This allows you to easily apply a pretrained
model to a new dataset and generate forecasts without the need to
retrain the model.

In [ ]:
ercot_df = pd.read_csv('https://datasets-nixtla.s3.amazonaws.com/ERCOT-clean.csv')

# we have to convert the ds column to integers
# since MLForecast was trained with that structure

ercot_df['ds'] = np.arange(1, len(ercot_df) + 1)
# use the `new_df` argument to pass the ercot dataset
ercot_fcsts = fcst.predict(horizon, new_df=ercot_df)
fig = plot_series(ercot_df, ercot_fcsts, max_insample_length=48 * 2)
fig

### Preprocess

If you want to take a look at the data that will be used to train the
models you can call `Forecast.preprocess`.

In [ ]:
prep_df = fcst.preprocess(train)
prep_df

If we do this we then have to call `Forecast.fit_models`, since this only stores the series information.

In [ ]:
X, y = prep_df.drop(columns=['unique_id', 'ds', 'y']), prep_df['y']
fcst.fit_models(X, y)

In [ ]:
fcst

In [ ]:
predictions2 = fcst.predict(horizon)
pd.testing.assert_frame_equal(predictions, predictions2)

If we would like to know how good our forecast will be for a specific
model and set of features then we can perform cross validation. What
cross validation does is take our data and split it in two parts, where
the first part is used for training and the second one for validation.
Since the data is time dependant we usually take the last *x*
observations from our data as the validation set.

This process is implemented in
[`MLForecast.cross_validation`](https://Nixtla.github.io/mlforecast/forecast.html#mlforecast.cross_validation),
which takes our data and performs the process described above for
`n_windows` times where each window has `h` validation samples in it.
For example, if we have 100 samples and we want to perform 2 backtests
each of size 14, the splits will be as follows:

1. Train: 1 to 72. Validation: 73 to 86.
2. Train: 1 to 86. Validation: 87 to 100.

You can control the size between each cross validation window using the
`step_size` argument. For example, if we have 100 samples and we want to
perform 2 backtests each of size 14 and move one step ahead in each fold
(`step_size=1`), the splits will be as follows:

1. Train: 1 to 85. Validation: 86 to 99.
2. Train: 1 to 86. Validation: 87 to 100.

You can also perform cross validation without refitting your models for
each window by setting `refit=False`. This allows you to evaluate the
performance of your models using multiple window sizes without having to
retrain them each time.

In [ ]:
fcst = MLForecast(
    models=lgb.LGBMRegressor(random_state=0, verbosity=-1),
    freq=1,
    lags=[24 * (i+1) for i in range(7)],
    lag_transforms={
        1: [RollingMean(window_size=24)],
        24: [RollingMean(window_size=24)],
        48: [ExponentiallyWeightedMean(alpha=0.3)],
    },
    num_threads=1,
    target_transforms=[Differences([24])],
)
cv_results = fcst.cross_validation(
    train,
    n_windows=2,
    h=horizon,
    step_size=horizon,
    fitted=True,
)
cv_results

Since we set `fitted=True` we can access the predictions for the
training sets as well with the `cross_validation_fitted_values` method.

In [ ]:
fcst.cross_validation_fitted_values()

We can also compute prediction intervals by passing a configuration to
`prediction_intervals` as well as values for the width through `levels`.

In [ ]:
cv_results_intervals = fcst.cross_validation(
    train,
    n_windows=2,
    h=horizon,
    step_size=horizon,
    prediction_intervals=PredictionIntervals(h=horizon),
    level=[80, 90]
)
cv_results_intervals

The `refit` argument allows us to control if we want to retrain the
models in every window. It can either be:

- A boolean: True will retrain on every window and False only on the first one.
- A positive integer: The models will be trained on the first window and then every `refit` windows.

In [ ]:
fcst = MLForecast(
    models=LinearRegression(),
    freq=1,
    lags=[1, 24],
)
for refit, expected_models in zip([True, False, 2], [4, 1, 2]):
    fcst.cross_validation(
        train,
        n_windows=4,
        h=horizon,
        refit=refit,
    )
assert len(fcst.cv_models_) == expected_models

In [ ]:
fig = plot_series(forecasts_df=cv_results.drop(columns='cutoff'))
fig

In [ ]:
fig = plot_series(forecasts_df=cv_results_intervals.drop(columns='cutoff'), level=[90])
fig

### Using LightGBMCV to tune your forecasts

Once you’ve found a set of features and parameters that work for your
problem you can build a forecast object from it using
[`MLForecast.from_cv`](https://Nixtla.github.io/mlforecast/forecast.html#mlforecast.from_cv),
which takes the trained
[`LightGBMCV`](https://Nixtla.github.io/mlforecast/lgb_cv.html#lightgbmcv)
object and builds an
[`MLForecast`](https://Nixtla.github.io/mlforecast/forecast.html#mlforecast)
object that will use the same features and parameters. Then you can call
fit and predict as you normally would.

In [ ]:
cv = LightGBMCV(
    freq=1,
    lags=[24 * (i+1) for i in range(7)],
    lag_transforms={
        48: [ExponentiallyWeightedMean(alpha=0.3)],
    },
    num_threads=1,
    target_transforms=[Differences([24])]
)
hist = cv.fit(
    train,
    n_windows=2,
    h=horizon,
    params={'verbosity': -1},
)

In [ ]:
fcst = MLForecast.from_cv(cv)
assert cv.best_iteration_ == fcst.models['LGBMRegressor'].n_estimators